# Library calling

In [1]:
#pip install selenium
#pip install webdriver_manager
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from urllib.parse import unquote
import pandas as pd
import datetime 

# Defining the Product Information and Location

In [2]:
SummaryFolder=r'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects'
summaryFile='Scraping_List.txt'
st=pd.read_csv(SummaryFolder+'\\'+summaryFile)
print(st)
#Define Product to extract
search_text = st['Product Name'][9]
print(search_text)
Source="CarParts"
OFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Outputs'
IFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Inputs'

                                  Product Name
0                                Ignition Coil
1                      Windshield Washer Pumps
2                        coupler trailer locks
3          Adjustable Trailer Hitch Ball Mount
4                               Vacuum Cleaner
5                                   Spark Plug
6              bluetooth Enabled Trailer locks
7                          Power Steering Hose
8   Power Steering Pressure Line Hose Assembly
9                       Washer Fluid Reservoir
10          Hitch Ball Mount with Weight Scale
11                               LED Headlamps
12                             LED flashlights
13                   Fiberglass Tonneau Covers
14                    Aluminium Tonneau Covers
15                     Hardfold Tonneau Covers
16                            Spark Plug Wires
17                 washer fluid reservoir tank
18                   Non Automotive Gas Struts
Washer Fluid Reservoir


# Setting Webdriver and Website Specific Information

In [3]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
driver.get('https://www.carparts.com/')
driver.maximize_window()
wait=WebDriverWait(driver, 5)

In [4]:
driver.find_element(By.CSS_SELECTOR,'[data-testid="desktop-geo-locator"]').click()
sleep(1)
pin = driver.find_element(By.CSS_SELECTOR, '[class="StyledTextInput-sc-1x30a0s-0 cZzNgu"]')
pin.click()
pin.clear()
pinnumber='94203'
pin.send_keys(pinnumber)

wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '[class="StyledButton-sc-323bzc-0 ignnRb"]'))).click()

In [5]:
# Find the search bar by its ID
try:
    search_bar = driver.find_element(By.ID, 'searchBox')
except:
    search_bar=driver.find_element(By.ID, "twotabsearchtextbox")

# Enter the search text

search_bar.send_keys(search_text)

# Submit the form to perform the search
search_bar.send_keys(Keys.ENTER)

In [12]:
#select 45 grid
page=driver.find_element(By.ID,"showItemsBox-2")
page.click()

# Defining the Dataframe and Extracting the data into the Dataframe

In [21]:
Cols=["Links"]
df = pd.DataFrame(columns=Cols)
Results=1
TotalResults= 2
count=0

In [22]:
while Results < 240:#TotalResults:
    sleep (2)
    pageNumbers=wait.until(EC.presence_of_element_located((By.ID,"topBarinformation"))).text
    Results= int(pageNumbers.split(" ")[3])
    TotalResults= pageNumbers.split(" ")[5]
    #print (Results, TotalResults)
    TP=int(TotalResults)
    PPP=int(Results)
    NoP=round(TP-PPP,0)
    print(PPP)
    for i in range(10):
        try:
            wait.until(EC.presence_of_element_located((By.TAG_NAME,'body'))).send_keys(Keys.END)
            sleep(1)
        except:
            break
    #Extracting hrefs
    element = driver.find_element(By.ID,"MainSection")
    anchor_elements = element.find_elements(By.TAG_NAME,'a')
    sleep(1)
    for anchor in anchor_elements:
        href = anchor.get_attribute('href')
        df.loc[count,'Links']=href
        count=count+1
    sleep(1)
    Nextpage=driver.find_element(By.ID,"pagination-next")
    Nextpage.click()

45
90
135
180
225
270


In [24]:
print(len(df))
df=df.drop_duplicates(["Links"])

270


# Post Processing Data and Exporting

In [25]:
df.to_excel(IFolder+'\\'+f'{Source}ProductLinks_'+search_text+'.xlsx', index=False)

# Archived Codes